# Integrating Multiple Datasets

In this tutorial, we will integrate two related samples together.



In [ ]:
import os
import platform
import random

# Make sure all the seeds are set
seed = 0
random.seed(seed)

# Only constrain threads on macOS where BLAS/numba deadlocks are common.
# Must run before any numpy/scanpy import.
if platform.system() == "Darwin":
    os.environ["OPENBLAS_NUM_THREADS"] = "1"
    os.environ["MKL_NUM_THREADS"] = "1"
    os.environ["NUMBA_NUM_THREADS"] = "1"

In [ ]:
import stlearn as st
import scanpy as sc
import harmonypy as hm
import pathlib
import numpy as np

st.settings.set_figure_params(dpi=120)
np.random.seed(seed)

## Read data

In this tutorial, we are using the Breast cancer datasets with 2 sections of block A.

Source:
 * https://www.10xgenomics.com/datasets/human-breast-cancer-block-a-section-1-1-standard-1-1-0 and
 * https://www.10xgenomics.com/datasets/human-breast-cancer-block-a-section-2-1-standard-1-1-0


In [ ]:
st.settings.datasetdir = pathlib.Path.cwd().parent / "data"

In [ ]:
sdata1 = st.datasets.visium_sge(sample_id="V1_Breast_Cancer_Block_A_Section_1")
sdata2 = st.datasets.visium_sge(sample_id="V1_Breast_Cancer_Block_A_Section_2")

In [ ]:
block1 = st.io.attach_spatial(
    sdata1.tables["table"],
    st.settings.datasetdir / "V1_Breast_Cancer_Block_A_Section_1" / "spatial",
    library_id="V1_Breast_Cancer_Block_A_Section_1",
)

In [ ]:
block2 = st.io.attach_spatial(
    sdata2.tables["table"],
    st.settings.datasetdir / "V1_Breast_Cancer_Block_A_Section_2" / "spatial",
    library_id="V1_Breast_Cancer_Block_A_Section_2",
)

## Processing data

In [ ]:
# concatenate 2 samples, with their images side by side for plotting
adata_concat = st.concatenate_spatial_adata([block1, block2])

In [ ]:
# Preprocessing
# Filter genes
sc.pp.filter_genes(adata_concat, min_cells=3)
# Normalize data
sc.pp.normalize_total(adata_concat, target_sum=1e4)
# Log transformation
sc.pp.log1p(adata_concat)
# Store raw data
adata_concat.raw = adata_concat
# Extract top highly variable genes
sc.pp.highly_variable_genes(adata_concat, min_mean=0.0125, max_mean=3, min_disp=0.5, batch_key="batch")
adata_concat = adata_concat[:, adata_concat.var.highly_variable].copy()
# Scale data
sc.pp.scale(adata_concat, max_value=10)

In [ ]:
# Run dimensionality reduction
sc.pp.pca(adata_concat, n_comps=30, svd_solver='arpack')

## Run integration with harmony

In [ ]:
# Prepare metadata and PCA
meta_data = adata_concat.obs
data_mat = adata_concat.obsm["X_pca"]

In [ ]:
# Run harmony
ho = hm.run_harmony(adata_concat.obsm["X_pca"], adata_concat.obs, "batch")

In [ ]:
# Mapping back the result to the adata object
Z = np.asarray(ho.Z_corr)
adata_concat.obsm["X_pca_harmony"] = Z if Z.shape[0] == adata_concat.n_obs else Z.T

## Perform clustering and visualize the results by UMAP

In [ ]:
# Build KNN and run UMAP
sc.pp.neighbors(adata_concat, n_pcs=30, use_rep="X_pca_harmony")
sc.tl.umap(adata_concat)

In [ ]:
# Run clustering with leiden
sc.tl.leiden(adata_concat, resolution=0.45, flavor="igraph", n_iterations=2, random_state=seed)

In [ ]:
# Plotting UMAP
sc.pl.umap(adata_concat, color=["batch", "leiden"])

## Map the result back to the original samples


In [ ]:
st.settings.set_figure_params(dpi=150)
# Map leiden clustering result to block A section 1
block1.obs["leiden"] = adata_concat.obs[adata_concat.obs.batch == "0"].leiden.values

In [ ]:
# Plotting the clustering result
st.pl.cluster_plot(block1, use_label="leiden")

In [ ]:
# Map leiden clustering result to block A section 2
block2.obs["leiden"] = adata_concat.obs[adata_concat.obs.batch == "1"].leiden.values

In [ ]:
# Plotting the clustering result
st.pl.cluster_plot(block2, use_label="leiden")

In [ ]:
st.settings.set_figure_params(dpi=200)
st.pl.gene_plot(adata_concat, gene_symbols="KRT5", use_raw=True, crop=False, size=1.4, cell_alpha=1)

In [ ]:
# Plot the clusters
st.pl.cluster_plot(adata_concat, use_label="leiden", crop=False, size=1.4, cell_alpha=1)